# Module 3: Classical Machine Learning - Data Preprocessing

In real-world machine learning, data is rarely clean and ready. It is often messy, has missing values, contains raw text or categories that algorithms cannot understand, and has attributes on completely different scales. 

**Data Preprocessing** is the pipeline of cleaning and transforming raw data into a clean matrix format that machine learning models can learn from. 

### Why this matters:
- **For Beginners**: Understanding this prevents errors like feeding raw string text into mathematical models, which causes crashes.
- **For Professionals**: Mastering preprocessing (specifically avoiding **Data Leakage**) is the difference between a model that works in production and one that fails when deployed.

In this notebook, we will cover:
1. **Loading Data & Exploration**: Inspecting values, shapes, types.
2. **Handling Missing Values**: Deleting vs. Imputing (filling in values).
3. **Encoding Categorical Data**: Representing text columns as numbers (Label vs. One-Hot encoding).
4. **Feature Scaling**: Adjusting scales (Normalization vs. Standardization).
5. **Dataset Splitting**: Separating Train and Test sets without data leakage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

---

## 1. Creating and Inspecting a Messy Dataset

Let's programmatically construct a mock dataset of house details to simulate real-world issues: missing values, text features, and varying numeric scales.

In [ ]:
data = {
    'Size_sqft': [1500, 2000, np.nan, 2400, 1800, 3000, 1200, np.nan, 2100, 1600],
    'Bedrooms': [3, 4, 3, np.nan, 3, 5, 2, 3, 4, 3],
    'Neighborhood': ['Suburbs', 'Downtown', 'Suburbs', 'Rural', 'Downtown', 'Downtown', 'Rural', 'Suburbs', np.nan, 'Suburbs'],
    'Price_USD': [250000, 400000, 280000, 310000, 370000, 550000, 180000, 290000, 330000, 260000]
}

df = pd.DataFrame(data)
print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())
print("\nData Types & Missing Counts:")
print(df.info())

---

## 2. Handling Missing Values

Many ML models cannot handle missing values (`NaN`). We have two primary strategies:
1. **Deletion**: Drop rows or columns with missing values. (Only recommended if missing values are extremely rare).
2. **Imputation**: Replace missing values with statistical measures (Mean, Median, Mode, or constant values).

In [ ]:
print("Missing values per column before:")
print(df.isnull().sum())

# Imputing Numerical values (using Median for robustness to outliers)
num_imputer = SimpleImputer(strategy='median')
df[['Size_sqft', 'Bedrooms']] = num_imputer.fit_transform(df[['Size_sqft', 'Bedrooms']])

# Imputing Categorical values (using Mode / most frequent value)
cat_imputer = SimpleImputer(strategy='most_frequent')
df[['Neighborhood']] = cat_imputer.fit_transform(df[['Neighborhood']])

print("\nMissing values after imputation:")
print(df.isnull().sum())
print("\nClean Dataframe:")
print(df.head())

---

## 3. Encoding Categorical Data

Models calculate using numbers. Strings like `'Downtown'` or `'Rural'` must be encoded.

### A. One-Hot Encoding (Nominal Categories)
For categorical variables without internal order (e.g. Neighborhoods), we split the column into multiple binary columns. This prevents the model from assuming `'Downtown' > 'Rural'` mathematically.

### B. Label Encoding (Ordinal Categories)
For variables with logical order (e.g. Ratings like `Low`, `Medium`, `High`), we map them to sequential numbers `0, 1, 2`.

In [ ]:
# One-Hot Encoding using pandas get_dummies (Simple approach)
df_encoded = pd.get_dummies(df, columns=['Neighborhood'], prefix='Loc', dtype=int)
print("One-hot Encoded Dataframe:")
print(df_encoded.head())

---

## 4. Feature Scaling

Compare the scales in our dataset: `Size_sqft` ranges from 1200 to 3000, while `Bedrooms` ranges from 2 to 5. Without scaling, gradient descent updates and distance calculations will be dominated by `Size_sqft` simply because its numbers are larger.

### A. Standardization (StandardScaler)
Centers the data around mean 0 with standard deviation 1. Highly recommended for algorithms assuming normal distribution (Linear/Logistic Regression, SVMs).
$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

### B. Normalization / MinMax Scaling (MinMaxScaler)
Scales all features strictly into a range of `[0, 1]`. Essential for Neural Networks and distance-based models (K-Means, KNN).
$$x_{\text{scaled}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}$$

In [ ]:
scaler = StandardScaler()

# Scale the numerical features
df_scaled = df_encoded.copy()
df_scaled[['Size_sqft', 'Bedrooms']] = scaler.fit_transform(df_scaled[['Size_sqft', 'Bedrooms']])

print("Scaled Dataframe:")
print(df_scaled.head())

---

## 5. Splitting the Dataset

To evaluate how well our model performs on unseen data, we divide our dataset into:
- **Training Set (typically 80%)**: Used to fit model parameters.
- **Testing Set (typically 20%)**: Held out, used strictly for scoring the final model.

> [!WARNING]
> **Data Leakage**: Always split your data **before** applying operations that compute dataset-wide statistics (like fit scaling or fit imputation) on the test set. Use `.fit_transform()` on the training set, but **only** `.transform()` on the testing set to avoid leaks!

In [ ]:
# Separate features (X) and target label (y)
X = df_encoded.drop(columns=['Price_USD'])
y = df_encoded['Price_USD']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape: {X_test.shape}")

# Correct Scaling workflow to prevent leakage:
scaler = StandardScaler()

# Fit & Transform training features
X_train_scaled = X_train.copy()
X_train_scaled[['Size_sqft', 'Bedrooms']] = scaler.fit_transform(X_train[['Size_sqft', 'Bedrooms']])

# ONLY Transform testing features
X_test_scaled = X_test.copy()
X_test_scaled[['Size_sqft', 'Bedrooms']] = scaler.transform(X_test[['Size_sqft', 'Bedrooms']])

---

## 🏋️ Exercises

### Exercise: Preprocess a Real-World Miniature Dataset
You are given a raw customer purchase log. Preprocess it by:
1. Imputing the missing values of `'Age'` with the column mean.
2. One-hot encoding the `'City'` column.
3. Min-Max scaling the numeric columns (`'Age'` and `'Salary'`).

In [ ]:
raw_data = {
    'Age': [25, 44, 30, 38, np.nan, 35, 48, 50],
    'City': ['New York', 'Paris', 'Paris', 'New York', 'Tokyo', 'Tokyo', 'New York', 'Paris'],
    'Salary': [50000, 80000, 62000, 72000, 95000, 110000, 85000, 90000]
}

df_exercise = pd.DataFrame(raw_data)

# YOUR CODE HERE